# Exact PPT-mixture certification of the asymmetric (3,3,2) GHZ state: reproducibility notebook

**Target:** \(|\psi\rangle=(|000\rangle+|111\rangle+|122\rangle)/\sqrt3\) in \(\mathbb{C}^2\otimes\mathbb{C}^3\otimes\mathbb{C}^3\), white noise \(\rho(p)=(1-p)|\psi\rangle\langle\psi|+p\,\mathbb{1}/18\).

This notebook is **generated** from the `gme332` package (`tools/build_notebook.py`); its code cell is identical to `src/gme332` and needs no installation of the package.

1. **Phase 0**: fidelity witness, \(\alpha=2/3\), \(p^*_F=6/17\).
2. **Phase 1**: partial-transpose spectra of the three cuts.
3. **Phase 2**: exact fully decomposable witness \(W_d\) (arithmetic over \(\mathbb{Q}(\sqrt{d-1})\)), exact PPT-mixture decomposition at \(d=3\) (threshold \(12/(16+\sqrt2)\)), numerical primal SDPs for \(d=2..6\) and block-reduced dual SDPs for \(d=2..8\).
4. **Phase 3**: Lindblad dynamics, one-sided \(dN/dt\) with the kernel term, root-found ESD times, \(N_g(t)\), GME lifetime and its bracket.
5. **Phase 3b**: \(\gamma_{\rm corr}\) sweep of the lifetimes.
6. **Phase 4**: local 5-qubit encoding, 5-CNOT circuit, FakeSherbrooke (IBM Eagle r3) calibration snapshot.
7. **Phase 5**: re-analysis of the photonic (3,3,2) experiment (Malik et al., Nat. Photon. 2016).

Every SDP checks the solver status and stops on failure.

**Run:** `Runtime -> Run all` (about 15 min). For a 1-2 min smoke test change the last line to `Settings.quick(fig_dir="./figures")`. Figures are saved to `./figures/`.


In [ ]:
# Dependencies (run once; ~2-3 min on a fresh Colab runtime).
!pip install -q "cvxpy>=1.5" "clarabel>=0.9" "qiskit>=2.0" "qiskit-aer>=0.17" "qiskit-ibm-runtime>=0.40" "qutip>=5.0" "numpy>=1.26" "scipy>=1.11" "matplotlib>=3.8"

import numpy, scipy, cvxpy, qiskit, qiskit_aer, qiskit_ibm_runtime, qutip, matplotlib
for mod in (numpy, scipy, cvxpy, qiskit, qiskit_aer, qiskit_ibm_runtime, qutip, matplotlib):
    print(f"{mod.__name__:20s}", mod.__version__)
print("\nIf an import failed: Runtime -> Restart runtime, then run this cell again.")

In [ ]:
# gme332 v1.1.0, generated by tools/build_notebook.py from src/gme332. Do not edit by hand.

# ============================================================================
# config.py
# ============================================================================
"""Global constants shared by all modules."""

RANDOM_SEED = 42
DIMS = [2, 3, 3]
CUTS = {"A|BC": (0,), "AB|C": (0, 1), "AC|B": (0, 2)}   # transposed side of each cut
SOLVER = "CLARABEL"

# ============================================================================
# basis.py
# ============================================================================
"""Basis indexing helpers (pure Python)."""
from typing import Sequence


def flat_index(s: Sequence[int], dims: Sequence[int]) -> int:
    idx = 0
    for k, dk in zip(s, dims):
        idx = idx * dk + k
    return idx

# ============================================================================
# exact.py
# ============================================================================
"""Exact arithmetic in Q(sqrt m) and exact positive-semidefiniteness tests.

Pure Python (fractions); NumPy is imported only by ExMat.to_numpy()."""
import itertools
import math
from fractions import Fraction
from typing import Dict



class QSqrt:
    """Exact number a + b*sqrt(m) with rational a, b (m a fixed positive integer)."""
    __slots__ = ("a", "b", "m")

    def __init__(self, a=0, b=0, m=2):
        self.a, self.b, self.m = Fraction(a), Fraction(b), int(m)
        r = math.isqrt(self.m)
        if r * r == self.m and self.b:          # sqrt(m) rational: keep a canonical form (b = 0)
            self.a, self.b = self.a + self.b * r, Fraction(0)

    def _c(self, o):
        return o if isinstance(o, QSqrt) else QSqrt(o, 0, self.m)

    def __add__(self, o):
        o = self._c(o); return QSqrt(self.a + o.a, self.b + o.b, self.m)
    __radd__ = __add__

    def __neg__(self):
        return QSqrt(-self.a, -self.b, self.m)

    def __sub__(self, o):
        return self + (-self._c(o))

    def __rsub__(self, o):
        return self._c(o) - self

    def __mul__(self, o):
        o = self._c(o)
        return QSqrt(self.a * o.a + self.m * self.b * o.b,
                     self.a * o.b + self.b * o.a, self.m)
    __rmul__ = __mul__

    def __truediv__(self, o):
        o = self._c(o)
        den = o.a * o.a - o.m * o.b * o.b
        return self * QSqrt(o.a / den, -o.b / den, self.m)

    def sign(self) -> int:
        a, b, m = self.a, self.b, self.m
        sa, sb = (a > 0) - (a < 0), (b > 0) - (b < 0)
        if sb == 0: return sa
        if sa == 0 or sa == sb: return sb if sa == 0 else sa
        return sa if a * a > m * b * b else sb

    def is_zero(self) -> bool:
        return self.a == 0 and self.b == 0

    def __float__(self):
        return float(self.a) + float(self.b) * math.sqrt(self.m)

    def __repr__(self):
        return f"{self.a} + {self.b}*sqrt({self.m})"


class ExMat:
    """Sparse exact symmetric matrix {(i, j): QSqrt} of size n (real entries)."""

    def __init__(self, n: int, m: int):
        self.n, self.m, self.e = n, m, {}

    def add(self, i, j, v):
        v = v if isinstance(v, QSqrt) else QSqrt(v, 0, self.m)
        self.e[(i, j)] = self.e.get((i, j), QSqrt(0, 0, self.m)) + v
        if self.e[(i, j)].is_zero():
            del self.e[(i, j)]
        return self

    def add_sym(self, i, j, v):
        self.add(i, j, v)
        return self if i == j else self.add(j, i, v)

    def add_outer(self, vec: Dict[int, object], w=1):
        for i, vi in vec.items():
            for j, vj in vec.items():
                self.add(i, j, QSqrt(w, 0, self.m) * vi * vj if not isinstance(w, QSqrt)
                         else w * vi * vj)
        return self

    def copy(self):
        out = ExMat(self.n, self.m); out.e = dict(self.e); return out

    def __add__(self, o):
        out = self.copy()
        for k, v in o.e.items(): out.add(*k, v)
        return out

    def __sub__(self, o):
        out = self.copy()
        for k, v in o.e.items(): out.add(*k, -v)
        return out

    def scale(self, c):
        out = ExMat(self.n, self.m)
        for k, v in self.e.items(): out.add(*k, v * c)
        return out

    def trace(self):
        return sum((v for (i, j), v in self.e.items() if i == j), QSqrt(0, 0, self.m))

    def inner(self, o):
        """Tr(self . o) for symmetric matrices."""
        return sum((v * o.e[(j, i)] for (i, j), v in self.e.items() if (j, i) in o.e),
                   QSqrt(0, 0, self.m))

    def equals(self, o) -> bool:
        return not (self - o).e

    def to_numpy(self):
        import numpy as np
        M = np.zeros((self.n, self.n))
        for (i, j), v in self.e.items(): M[i, j] = float(v)
        return M


def ex_partial_transpose(M: ExMat, dims, parties) -> ExMat:
    dims = list(dims)
    def unflat(k):
        out = []
        for dk in reversed(dims):
            out.append(k % dk); k //= dk
        return out[::-1]
    out = ExMat(M.n, M.m)
    for (i, j), v in M.e.items():
        r, c = unflat(i), unflat(j)
        for p in parties: r[p], c[p] = c[p], r[p]
        out.add(flat_index(r, dims), flat_index(c, dims), v)
    return out


def ex_swap_BC(M: ExMat, d: int) -> ExMat:
    dims = [2, d, d]
    def sw(k):
        a, rem = divmod(k, d * d); b, c = divmod(rem, d)
        return flat_index((a, c, b), dims)
    out = ExMat(M.n, M.m)
    for (i, j), v in M.e.items(): out.add(sw(i), sw(j), v)
    return out


def _ex_det(B):
    n = len(B)
    if n == 1: return B[0][0]
    tot = QSqrt(0, 0, B[0][0].m)
    for c in range(n):
        if B[0][c].is_zero(): continue
        minor = [row[:c] + row[c + 1:] for row in B[1:]]
        term = B[0][c] * _ex_det(minor)
        tot = tot + term if c % 2 == 0 else tot - term
    return tot


def ex_is_psd(M: ExMat) -> bool:
    """Exact PSD test: on every connected block, all elementary symmetric
    functions of the eigenvalues (sums of k x k principal minors) are >= 0."""
    adj = {i: set() for i in range(M.n)}
    for (i, j) in M.e: adj[i].add(j); adj[j].add(i)
    seen = set()
    for s in range(M.n):
        if s in seen: continue
        comp, stack = [], [s]
        while stack:
            k = stack.pop()
            if k in seen: continue
            seen.add(k); comp.append(k); stack += list(adj[k] - seen)
        comp.sort()
        zero = QSqrt(0, 0, M.m)
        B = [[M.e.get((i, j), zero) for j in comp] for i in comp]
        for k in range(1, len(comp) + 1):
            ek = zero
            for sub in itertools.combinations(range(len(comp)), k):
                ek = ek + _ex_det([[B[i][j] for j in sub] for i in sub])
            if ek.sign() < 0:
                return False
    return True

# ============================================================================
# states.py
# ============================================================================
"""States, partial transpose and negativity (numerical, NumPy)."""
from typing import Sequence

import numpy as np



def ghz_family_ket(d: int) -> np.ndarray:
    """|psi_d> = (|000> + sum_{j=1}^{d-1} |1jj>) / sqrt(d)  in C^2 (x) C^d (x) C^d."""
    dims = [2, d, d]
    ket  = np.zeros(2 * d * d)
    ket[flat_index((0, 0, 0), dims)] = 1.0
    for j in range(1, d):
        ket[flat_index((1, j, j), dims)] = 1.0
    return ket / np.linalg.norm(ket)


def isotropic_noise(rho_pure: np.ndarray, p: float) -> np.ndarray:
    D = rho_pure.shape[0]
    return (1 - p) * rho_pure + p * np.eye(D) / D


def partial_transpose(rho: np.ndarray, dims: Sequence[int],
                      parties: Sequence[int]) -> np.ndarray:
    """Reshape to (row_0..row_{N-1}, col_0..col_{N-1}) and swap the row/col
    axes of every transposed party (valid for unequal local dimensions)."""
    dims = list(dims); N = len(dims); D = int(np.prod(dims))
    t = rho.reshape(dims + dims)
    for p in parties:
        t = np.swapaxes(t, p, p + N)
    return t.reshape(D, D)


def negativity(rho: np.ndarray, dims, parties) -> float:
    ev = np.linalg.eigvalsh(partial_transpose(rho, dims, parties))
    return float(-ev[ev < 0].sum())


def negativity_rate_terms(rho, drho, dims, parties, tol=1e-9):
    """The two parts of the one-sided derivative of N(rho + t drho) at t = 0+:
    (negative-space term -Tr(Pi_- X'), kernel term sum_{mu<0}|mu| over the
    spectrum of Pi_0 X' Pi_0).  Degenerate first-order perturbation theory."""
    X  = partial_transpose(rho, dims, parties)
    dX = partial_transpose(drho, dims, parties)
    ev, V = np.linalg.eigh(X)
    Vn = V[:, ev < -tol]; V0 = V[:, np.abs(ev) <= tol]
    neg = -float(np.real(np.trace(Vn.conj().T @ dX @ Vn)))
    ker = 0.0
    if V0.shape[1]:
        mu = np.linalg.eigvalsh(V0.conj().T @ dX @ V0)
        ker = -float(mu[mu < 0].sum())
    return neg, ker


def negativity_right_derivative(rho, drho, dims, parties, tol=1e-9) -> float:
    """One-sided derivative d/dt N(rho + t drho) at t = 0+.
    Negative eigenspace contributes -Tr(Pi_- X'), and eigenvalues in the kernel
    of T(rho) that are pushed below zero contribute |mu| (degenerate
    first-order perturbation theory on Pi_0 X' Pi_0)."""
    X  = partial_transpose(rho, dims, parties)
    dX = partial_transpose(drho, dims, parties)
    ev, V = np.linalg.eigh(X)
    Vn = V[:, ev < -tol]; V0 = V[:, np.abs(ev) <= tol]
    rate = -float(np.real(np.trace(Vn.conj().T @ dX @ Vn)))
    if V0.shape[1]:
        mu = np.linalg.eigvalsh(V0.conj().T @ dX @ V0)
        rate += -float(mu[mu < 0].sum())
    return rate

# ============================================================================
# sdp.py
# ============================================================================
"""Numerical SDPs (CVXPY + CLARABEL): PPT-mixture threshold, decomposable
witness value, genuine multipartite negativity, symmetry-reduced family SDPs
(primal and dual)."""
import itertools
import sys
import warnings
from typing import Dict, List, Tuple

import cvxpy as cp
import numpy as np



class SolverError(RuntimeError):
    """Raised when an SDP does not reach an optimal status."""


def _solve(prob: cp.Problem) -> float:
    """Solve with the configured solver and check the status: 'optimal' is
    accepted, 'optimal_inaccurate' is accepted with a RuntimeWarning, anything
    else raises SolverError."""
    caller = sys._getframe(1).f_code.co_name
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)     # CVXPY's own message; reported below with context
        prob.solve(solver=SOLVER)
    if prob.status == cp.OPTIMAL:
        return float(prob.value)
    if prob.status == cp.OPTIMAL_INACCURATE:
        warnings.warn(f"{caller}: SDP solved only to reduced accuracy (status {prob.status}, "
                      f"value {prob.value:.3e})", RuntimeWarning, stacklevel=3)
        return float(prob.value)
    raise SolverError(f"{caller}: SDP failed with status {prob.status!r}")


def _pt(X, dims, parties):
    for q in parties:
        X = cp.partial_transpose(X, dims, q)
    return X


def _var(D, real):
    return cp.Variable((D, D), symmetric=True) if real else cp.Variable((D, D), hermitian=True)


def _objective(W, rho, real):
    """Tr(W rho) as a real affine expression. With real=True the state must be
    real (up to rounding); its real part is used, so real- and complex-dtype
    inputs are both accepted."""
    rho = np.asarray(rho)
    if real:
        if np.iscomplexobj(rho) and np.abs(rho.imag).max() > 1e-12:
            raise ValueError("real=True requires a real density matrix")
        return cp.trace(W @ np.real(rho))
    return cp.real(cp.trace(W @ rho))


def ppt_mixture_noise_threshold(rho, dims, cuts=CUTS, real=False) -> float:
    """Smallest white-noise weight q such that (1-q) rho + q 1/D is a PPT mixture."""
    D = rho.shape[0]
    if real:
        rho = np.real(rho)
    q = cp.Variable(); parts = [_var(D, real) for _ in cuts]
    cons = [(1 - q) * rho + q * np.eye(D) / D == sum(parts)]
    for X, k in zip(parts, cuts.values()):
        cons += [X >> 0, _pt(X, dims, k) >> 0]
    return _solve(cp.Problem(cp.Minimize(q), cons))


def decomposable_witness_value(rho, dims, cuts=CUTS, real=False) -> Tuple[float, np.ndarray]:
    """min Tr(W rho) over fully decomposable W (W = P_k + T_k(Q_k), P_k, Q_k >= 0)
    with Tr W = 1.  Negative  <=>  rho is not a PPT mixture  =>  rho is GME."""
    D = rho.shape[0]
    W = _var(D, real); cons = [(cp.trace(W) if real else cp.real(cp.trace(W))) == 1]
    for k in cuts.values():
        P, Q = _var(D, real), _var(D, real)
        cons += [P >> 0, Q >> 0, W == P + _pt(Q, dims, k)]
    value = _solve(cp.Problem(cp.Minimize(_objective(W, rho, real)), cons))
    return value, W.value


def genuine_multipartite_negativity(rho, dims, cuts=CUTS, real=False) -> float:
    """N_g(rho) = -min Tr(W rho), W = P_k + T_k(Q_k), 0 <= P_k, Q_k <= 1
    [Jungnitsch, Moroder, Guehne 2011].  N_g > 0  <=>  not a PPT mixture.
    Small negative optimal values (solver noise around zero) are reported as 0."""
    D = rho.shape[0]; I = np.eye(D)
    W = _var(D, real); cons = []
    for k in cuts.values():
        P, Q = _var(D, real), _var(D, real)
        cons += [P >> 0, Q >> 0, P << I, Q << I, W == P + _pt(Q, dims, k)]
    value = _solve(cp.Problem(cp.Minimize(_objective(W, rho, real)), cons))
    if value > 1e-7:
        warnings.warn(f"positive optimal value {value:.2e} in N_g (should be <= 0)", RuntimeWarning)
    return max(0.0, -value)


def _torus_charge(a, b, c, d):
    """Charge of |abc> under local diagonal unitaries that fix |psi_d>
    (free phases: alpha, beta_0..beta_{d-1}, gamma_1)."""
    v = np.zeros(d + 2, dtype=int)
    v[0] += a; v[1 + b] += 1
    if c == 0:   v[0] += 1; v[1] -= 1; v[2] += 1; v[-1] += 1
    elif c == 1: v[-1] += 1
    else:        v[2] += 1; v[1 + c] -= 1; v[-1] += 1
    return tuple(v)


def family_threshold_sdp(d: int) -> float:
    """PPT-mixture threshold for |psi_d>, reduced to real torus-invariant
    operators (twirling a feasible point keeps it feasible)."""
    dims = [2, d, d]; D = 2 * d * d
    basis = list(itertools.product(range(2), range(d), range(d)))
    ch = [_torus_charge(*s, d) for s in basis]
    rho = np.outer(ghz_family_ket(d), ghz_family_ket(d))
    q = cp.Variable(); parts = []; cons = []
    for k in CUTS.values():
        X = cp.Variable((D, D), symmetric=True)
        cons += [X[i, j] == 0 for i in range(D) for j in range(i, D) if ch[i] != ch[j]]
        cons += [X >> 0, _pt(X, dims, k) >> 0]; parts.append(X)
    cons += [(1 - q) * rho + q * np.eye(D) / D == sum(parts)]
    return _solve(cp.Problem(cp.Minimize(q), cons))


# ---------------------------------------------------------------------------
# Block-reduced dual SDP for the 2 x d x d family
# ---------------------------------------------------------------------------

def _blocks(mask: np.ndarray) -> List[List[int]]:
    """Connected components of the sparsity graph of a D x D boolean mask."""
    D = mask.shape[0]; adj = mask | mask.T; seen = np.zeros(D, bool); out = []
    for s in range(D):
        if seen[s]:
            continue
        stack, comp = [s], []
        while stack:
            k = stack.pop()
            if seen[k]:
                continue
            seen[k] = True; comp.append(k); stack += list(np.nonzero(adj[k])[0])
        out.append(sorted(comp))
    return out


def _pt_permutation(basis, parties) -> np.ndarray:
    """perm such that vec(T(X)) = vec(X)[perm] (row-major vec, D x D)."""
    flat = {s: i for i, s in enumerate(basis)}; D = len(basis)
    perm = np.zeros(D * D, dtype=int)
    for i, s in enumerate(basis):
        for j, t in enumerate(basis):
            a, b = list(s), list(t)
            for p in parties:
                a[p], b[p] = b[p], a[p]
            perm[flat[tuple(a)] * D + flat[tuple(b)]] = i * D + j
    return perm


def family_witness_sdp(d: int) -> Dict:
    """Dual of family_threshold_sdp: min Tr(W rho_d) over fully decomposable W
    with Tr W = 1, reduced by symmetry.  Twirling over the torus keeps P_k and
    R_k := T_k(Q_k) invariant, so both are parametrised in the invariant
    pattern, Q_k = T_k(R_k) >= 0 is imposed, and every PSD constraint is split
    into the connected blocks of its sparsity pattern (block size <= d).
    Returns c_min and the threshold |c|/(|c| + 1/D) it implies."""
    D = 2 * d * d
    basis = list(itertools.product(range(2), range(d), range(d)))
    ch = [_torus_charge(*s, d) for s in basis]
    allowed = [(i, j) for i in range(D) for j in range(i, D) if ch[i] == ch[j]]
    B = np.zeros((D * D, len(allowed)))
    for k, (i, j) in enumerate(allowed):
        B[i * D + j, k] = 1; B[j * D + i, k] = 1
    mask = (B.sum(axis=1) > 0).reshape(D, D)

    def psd(vec, pattern):
        M = cp.reshape(vec, (D, D), order="C")
        return [M[np.ix_(c, c)] >> 0 if len(c) > 1 else M[c[0], c[0]] >= 0 for c in _blocks(pattern)]

    cons, W = [], None
    for parties in CUTS.values():
        perm = _pt_permutation(basis, parties)
        Pv = B @ cp.Variable(len(allowed)); Rv = B @ cp.Variable(len(allowed))
        cons += psd(Pv, mask) + psd(Rv[perm], mask.reshape(-1)[perm].reshape(D, D))
        if W is None:
            W = Pv + Rv
        else:
            cons.append(Pv + Rv == W)
    rho = np.outer(ghz_family_ket(d), ghz_family_ket(d)).reshape(-1)
    cons.append(cp.trace(cp.reshape(W, (D, D), order="C")) == 1)
    c = _solve(cp.Problem(cp.Minimize(rho @ W), cons))
    return dict(c_min=c, threshold=abs(c) / (abs(c) + 1.0 / D))


def weak_constraint_pitfall(rho, dims) -> Dict:
    """Solve min Tr(W rho) s.t. Tr W = 1, W + T_k(W) >= 0 (a constraint that is
    NOT sufficient for a witness) and evaluate the optimiser on the product state
    (|0>+i|1>)/sqrt2 (x) (|00> - i|11> - i|22>)/sqrt3."""
    D = rho.shape[0]
    W = cp.Variable((D, D), symmetric=True)
    cons = [cp.trace(W) == 1] + [W + _pt(W, dims, k) >> 0 for k in CUTS.values()]
    value = _solve(cp.Problem(cp.Minimize(cp.trace(W @ np.real(rho))), cons))
    a   = np.array([1, 1j]) / np.sqrt(2)
    phi = np.zeros(9, complex); phi[0] = 1; phi[4] = -1j; phi[8] = -1j; phi /= np.sqrt(3)
    sig = np.kron(a, phi)
    return dict(value=value,
                naive_threshold=abs(value) / (abs(value) + 1 / D),
                value_on_product_state=float(np.real(sig.conj() @ W.value @ sig)))

# ============================================================================
# dynamics.py
# ============================================================================
"""Lindblad dynamics: Liouvillian, exact propagator, ESD times, GME lifetime."""
from typing import List

import numpy as np
import scipy.linalg as la
from scipy.optimize import brentq



def _embed(op, party, dims):
    ops = [np.eye(d) for d in dims]; ops[party] = op
    out = ops[0]
    for o in ops[1:]: out = np.kron(out, o)
    return out


def _z_centred(d):
    z = np.arange(d, dtype=float) - (d - 1) / 2.0
    return np.diag(z / np.linalg.norm(z))


def collapse_operators(dims, gamma_1, gamma_phi, gamma_corr) -> List[np.ndarray]:
    ops = []
    for p, d in enumerate(dims):
        ops.append(np.sqrt(gamma_1) * _embed(np.diag(np.sqrt(np.arange(1, d)), 1), p, dims))
        ops.append(np.sqrt(gamma_phi) * _embed(_z_centred(d), p, dims))
    if gamma_corr > 0:
        ops.append(np.sqrt(gamma_corr) * np.kron(np.kron(np.eye(dims[0]), _z_centred(dims[1])),
                                                 _z_centred(dims[2])))
    return ops


def liouvillian(dims, gamma_1, gamma_phi, gamma_corr) -> np.ndarray:
    """Row-major superoperator: vec(C rho C^+) = (C (x) C*) vec(rho)."""
    D = int(np.prod(dims)); I = np.eye(D); L = np.zeros((D * D, D * D), complex)
    for C in collapse_operators(dims, gamma_1, gamma_phi, gamma_corr):
        CdC = C.conj().T @ C
        L += np.kron(C, C.conj()) - 0.5 * np.kron(CdC, I) - 0.5 * np.kron(I, CdC.T)
    return L


class Propagator:
    """Exact solution rho(t) = exp(L t) rho(0) via the eigendecomposition of L,
    falling back to expm if L is numerically defective."""

    def __init__(self, L, rho0):
        self.L, self.v0, self.D = L, rho0.reshape(-1), rho0.shape[0]
        w, V = np.linalg.eig(L)
        self.ok = np.linalg.cond(V) < 1e8
        if self.ok:
            self.w, self.V, self.c = w, V, np.linalg.solve(V, self.v0)

    def __call__(self, t):
        if self.ok:
            v = self.V @ (np.exp(self.w * t) * self.c)
        else:
            v = la.expm(self.L * t) @ self.v0
        r = v.reshape(self.D, self.D)
        return (r + r.conj().T) / 2


def esd_time(prop, dims, parties, t_max, n_grid=301) -> float:
    """Exact ESD time: first root of lambda_min(T_k rho(t)) = 0."""
    f = lambda t: np.linalg.eigvalsh(partial_transpose(prop(t), dims, parties))[0]
    ts = np.linspace(0, t_max, n_grid)
    vals = [f(t) for t in ts]
    for i in range(1, n_grid):
        if vals[i] >= 0 > vals[i - 1]:
            return brentq(f, ts[i - 1], ts[i], xtol=1e-10)
    return float("nan")


def gme_lifetime(prop, dims, t_hi, xtol=1e-4) -> float:
    """First time rho(t) becomes a PPT mixture (decomposable-witness value = 0).
    t_hi must be a time at which rho is already PPT across some cut."""
    f = lambda t: decomposable_witness_value(prop(t), dims, real=True)[0]
    return brentq(f, 0.0, t_hi, xtol=xtol)

# ============================================================================
# certificates.py
# ============================================================================
"""Exact certificates: the fully decomposable witness family W_d and the
exact PPT-mixture decomposition of rho(p*) at d = 3."""
from fractions import Fraction
from typing import Dict



def exact_witness_family(d: int) -> Dict:
    """Unnormalised fully decomposable witness W~_d for |psi_d> and, for every
    cut k, its decomposition W~_d = P_k + T_k(Q_k) with P_k, Q_k >= 0.
    lam = 1/sqrt(d-1); all entries lie in Q(sqrt(d-1))."""
    m    = d - 1
    dims = [2, d, d]; D = 2 * d * d
    ix   = lambda a, b, c: flat_index((a, b, c), dims)
    one  = QSqrt(1, 0, m)
    rt   = QSqrt(0, 1, m)                       # sqrt(m)
    lam  = QSqrt(0, Fraction(1, m), m)          # 1/sqrt(m)
    js   = range(1, d)
    ghz0 = {ix(0, j, j): one for j in js}       # sum_j |0jj>

    W = ExMat(D, m)
    for j in js:
        W.add_sym(ix(0, 0, 0), ix(1, j, j), -one)
        for s in (ix(0, 0, j), ix(0, j, 0), ix(1, 0, j), ix(1, j, 0)):
            W.add(s, s, one)
    W.add_outer(ghz0, lam)
    W.add(ix(1, 0, 0), ix(1, 0, 0), rt)

    # cut A|BC
    P1 = ExMat(D, m)
    for j in js:
        for s in (ix(0, 0, j), ix(0, j, 0), ix(1, 0, j), ix(1, j, 0)):
            P1.add(s, s, one)
    u = dict(ghz0); u[ix(1, 0, 0)] = -rt
    Q1 = ExMat(D, m).add_outer(u, lam)
    # cut AB|C  (and AC|B by B<->C symmetry)
    def cut_pair(pairs, diag):
        P = ExMat(D, m).add_outer(ghz0, lam)
        P.add(ix(1, 0, 0), ix(1, 0, 0), rt)
        for s in diag: P.add(s, s, one)
        Q = ExMat(D, m)
        for a, b in pairs: Q.add_outer({a: one, b: -one})
        return P, Q
    P2, Q2 = cut_pair([(ix(0, 0, j), ix(1, j, 0)) for j in js],
                      [s for j in js for s in (ix(0, j, 0), ix(1, 0, j))])
    P3, Q3 = cut_pair([(ix(0, j, 0), ix(1, 0, j)) for j in js],
                      [s for j in js for s in (ix(0, 0, j), ix(1, j, 0))])
    return dict(d=d, m=m, dims=dims, D=D, W=W,
                decomposition={"A|BC": (P1, Q1), "AB|C": (P2, Q2), "AC|B": (P3, Q3)})


def verify_witness_family(d: int) -> Dict:
    """Exact checks: W~ = P_k + T_k(Q_k), P_k, Q_k >= 0 for all k; Tr W~; <psi~|W~|psi~>;
    threshold p*(d) at which Tr(W~ rho(p)) = 0."""
    wf = exact_witness_family(d)
    W, dims, D, m = wf["W"], wf["dims"], wf["D"], wf["m"]
    ok_decomp, ok_psd = True, True
    for name, (P, Q) in wf["decomposition"].items():
        ok_decomp &= (P + ex_partial_transpose(Q, dims, CUTS[name])).equals(W)
        ok_psd    &= ex_is_psd(P) and ex_is_psd(Q)
    psi_t = ExMat(D, m).add_outer({flat_index((0, 0, 0), dims): QSqrt(1, 0, m),
                                   **{flat_index((1, j, j), dims): QSqrt(1, 0, m)
                                      for j in range(1, d)}})
    trW      = W.trace()                          # 4(d-1) + 2 sqrt(d-1)
    wpsi     = W.inner(psi_t) / d                 # Tr(W~ rho_psi) = -2(d-1)/d
    p_star   = (-wpsi) / ((-wpsi) + trW / D)      # root of (1-p) wpsi + p trW / D
    return dict(d=d, decomposition_exact=ok_decomp, P_Q_psd_exact=ok_psd,
                trace_W=trW, tr_W_rho_psi=wpsi, c_normalised=wpsi / trW,
                p_star=p_star, W_numeric=W.to_numpy() / float(trW))


def exact_primal_certificate_d3() -> Dict:
    """Exact PPT-mixture decomposition rho(p*) = rho_1 + rho_2 + rho_3 at
    p* = 12/(16+sqrt2), rho_k >= 0 and T_k(rho_k) >= 0.  rho_2 is listed
    explicitly, rho_3 = S rho_2 S (S swaps B and C), rho_1 = rho(p*) - rho_2 - rho_3.
    Entries lie in Q(sqrt2); n = p*/18 = (16 - sqrt2)/381."""
    d, dims, D = 3, [2, 3, 3], 18
    ix  = lambda s: flat_index(tuple(int(ch) for ch in s), dims)
    R   = lambda num, den: QSqrt(Fraction(num, den), 0, 2)
    p_star = QSqrt(12, 0, 2) / QSqrt(16, 1, 2)
    n   = p_star / 18
    mm  = (1 - p_star) / 3
    rho_p = ExMat(D, 2)
    for k in range(D): rho_p.add(k, k, n)
    for a in ("000", "111", "122"):
        for b in ("000", "111", "122"): rho_p.add(ix(a), ix(b), mm)

    rho2 = ExMat(D, 2)
    entries = [("000", "000", R(13, 200)), ("000", "111", n), ("000", "122", n),
               ("001", "001", n), ("002", "002", n), ("110", "110", n), ("120", "120", n),
               ("011", "011", R(7, 1000)), ("022", "022", R(7, 1000)),
               ("011", "022", R(-7, 1000)),
               ("012", "012", R(13, 1000)), ("021", "021", R(13, 1000)),
               ("111", "111", R(31, 1000)), ("122", "122", R(31, 1000)),
               ("111", "122", R(3, 200)),
               ("112", "112", R(2, 125)), ("121", "121", R(2, 125))]
    for a, b, v in entries: rho2.add_sym(ix(a), ix(b), v)
    rho3 = ex_swap_BC(rho2, d)
    rho1 = rho_p - rho2 - rho3
    parts = {"A|BC": rho1, "AB|C": rho2, "AC|B": rho3}
    checks = {}
    for name, rk in parts.items():
        checks[name] = dict(rho_psd=ex_is_psd(rk),
                            pt_psd=ex_is_psd(ex_partial_transpose(rk, dims, CUTS[name])))
    # complementary slackness with the witness decomposition (optimality pairing)
    wf = exact_witness_family(3)
    slack = {name: (P.inner(parts[name]),
                    Q.inner(ex_partial_transpose(parts[name], dims, CUTS[name])))
             for name, (P, Q) in wf["decomposition"].items()}
    slack_zero = all(a.is_zero() and b.is_zero() for a, b in slack.values())
    sums_ok = (rho1 + rho2 + rho3).equals(rho_p)
    return dict(p_star=p_star, checks=checks, sum_equals_rho=sums_ok,
                complementary_slackness=slack_zero,
                all_ok=sums_ok and all(c["rho_psd"] and c["pt_psd"] for c in checks.values()),
                rho_parts={k: v.to_numpy() for k, v in parts.items()})

# ============================================================================
# experiment.py
# ============================================================================
_MALIK2016_CSV = '# Measured density-matrix elements of the (3,3,2) three-photon state.\n# Source: M. Malik et al., "Multi-photon entanglement in high dimensions",\n#         Nat. Photon. 10, 248-252 (2016), arXiv:1509.02561, Extended Data Table 1.\n# Labels are the authors\' OAM labels: A, B in {-1,0,1}, C in {0,1}.\n# kind = pop (diagonal element <ABC|rho|ABC>) or coh (off-diagonal <row|rho|col>).\nkind,row_A,row_B,row_C,col_A,col_B,col_C,re,im\npop,-1,-1,0,-1,-1,0,0.003,0\npop,0,-1,0,0,-1,0,0.009,0\npop,1,-1,0,1,-1,0,0.003,0\npop,-1,0,0,-1,0,0,0.006,0\npop,0,0,0,0,0,0,0.296,0\npop,1,0,0,1,0,0,0.011,0\npop,-1,1,0,-1,1,0,0.000,0\npop,0,1,0,0,1,0,0.014,0\npop,1,1,0,1,1,0,0.003,0\npop,-1,-1,1,-1,-1,1,0.006,0\npop,0,-1,1,0,-1,1,0.017,0\npop,1,-1,1,1,-1,1,0.290,0\npop,-1,0,1,-1,0,1,0.006,0\npop,0,0,1,0,0,1,0.003,0\npop,1,0,1,1,0,1,0.003,0\npop,-1,1,1,-1,1,1,0.313,0\npop,0,1,1,0,1,1,0.014,0\npop,1,1,1,1,1,1,0.003,0\ncoh,0,0,0,1,-1,1,0.190,-0.133\ncoh,0,0,0,-1,1,1,0.178,-0.129\ncoh,1,-1,1,-1,1,1,0.297,-0.026\n'

"""Re-analysis of the photonic (3,3,2) experiment of Malik et al.,
Nat. Photon. 10, 248 (2016), with the exact witness W3.

The authors' OAM labels (A, B in {-1,0,1}, C in {0,1}) map to this package's
labels |a b c> by  a = C,  b = {0:0, 1:1, -1:2}[A],  c = {0:0, -1:1, 1:2}[B].
"""
import csv
import io
from importlib import resources
from typing import Dict, Tuple

import numpy as np

DATA_FILE = "malik2016_extended_data_table1.csv"


def load_malik2016() -> Tuple[Dict, Dict]:
    """Return (populations, coherences) keyed by the authors' (A, B, C) labels."""
    text = _MALIK2016_CSV
    rows = csv.DictReader(io.StringIO("".join(l for l in text.splitlines(True)
                                              if not l.startswith("#"))))
    pops, cohs = {}, {}
    for r in rows:
        row = (int(r["row_A"]), int(r["row_B"]), int(r["row_C"]))
        col = (int(r["col_A"]), int(r["col_B"]), int(r["col_C"]))
        val = complex(float(r["re"]), float(r["im"]))
        if r["kind"] == "pop":
            pops[row] = val.real
        else:
            cohs[(row, col)] = val
    return pops, cohs


def malik2016_certificate() -> Dict:
    """Upper bound on Tr(W3 rho_exp) from the published elements; the unmeasured
    coherence rho_{011,022} is bounded by positivity, |rho_ij| <= sqrt(rho_ii rho_jj).
    Local phases are aligned by a diagonal unitary on party B (keeps W3 fully decomposable)."""
    populations, coherences = load_malik2016()
    fA, fB = {0: 0, 1: 1, -1: 2}, {0: 0, -1: 1, 1: 2}
    lab = lambda t: f"{t[2]}{fA[t[0]]}{fB[t[1]]}"
    pop = {lab(k): v for k, v in populations.items()}
    coh = {(lab(a), lab(b)): v for (a, b), v in coherences.items()}
    z1, z2, z3 = coh[("000", "111")], coh[("000", "122")], coh[("111", "122")]
    c1, c2 = abs(z1), abs(z2)
    # Local diagonal unitaries can rotate rho_{000,111} and rho_{000,122} onto the
    # positive real axis; the phase of rho_{111,122} is then fixed:
    # arg z3' = arg z3 + arg z1 - arg z2.
    z3_aligned = z3 * (z1 / c1) * np.conj(z2 / c2)
    F = (pop["000"] + pop["111"] + pop["122"] + 2 * (c1 + c2 + z3_aligned.real)) / 3
    S8 = ["001", "002", "010", "020", "101", "102", "110", "120"]
    g_term = (np.sqrt(pop["011"]) + np.sqrt(pop["022"])) ** 2 / np.sqrt(2)
    W_t = -2 * (c1 + c2) + sum(pop[s] for s in S8) + g_term + np.sqrt(2) * pop["100"]
    trW = 8 + 2 * np.sqrt(2)

    def worst_case(delta):
        """W~ bound if every element entering it is shifted adversarially by delta."""
        g_hi = (np.sqrt(pop["011"] + delta) + np.sqrt(pop["022"] + delta)) ** 2 / np.sqrt(2)
        return W_t + 2 * 2 * delta + len(S8) * delta + np.sqrt(2) * delta + (g_hi - g_term)

    return dict(F=F, witness_unnorm=W_t, witness=W_t / trW,
                q_witness=-W_t / (-W_t + trW / 18),
                q_fidelity=(F - 2 / 3) / (F - 1 / 18),
                coherence_moduli=(c1, c2),
                coherence_phases_deg=tuple(float(np.degrees(np.angle(z))) for z in (z1, z2, z3)),
                residual_phase_deg=float(np.degrees(np.angle(z3_aligned))),
                g_term_bound=g_term,
                witness_worst_case={dl: worst_case(dl) / trW for dl in (0.0005, 0.005, 0.01)},
                coherence_sum_needed=(W_t + 2 * (c1 + c2)) / 2, coherence_sum=c1 + c2)

# ============================================================================
# device.py
# ============================================================================
"""Noisy-device simulation: local qubit encoding, 5-CNOT circuit,
FakeSherbrooke (IBM Eagle r3) calibration snapshot, density-matrix GME tests."""
import itertools
from typing import Dict

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke



def encoded_register_index(a, b, c):
    """Little-endian register index for the local encoding A->q0, B->(q1,q2),
    C->(q3,q4); qutrit level j is the two-qubit state |j> (j = 0, 1, 2)."""
    return a + 2 * b + 8 * c


def register_to_party_order(rho_reg):
    """Reorder the 32 x 32 register density matrix to the party basis
    |a, b4, c4> of C^2 (x) C^4 (x) C^4 (index 16 a + 4 b4 + c4)."""
    perm = np.zeros(32, dtype=int)
    for a, b4, c4 in itertools.product(range(2), range(4), range(4)):
        perm[16 * a + 4 * b4 + c4] = a + 2 * b4 + 8 * c4
    return rho_reg[np.ix_(perm, perm)]


def ghz233_circuit() -> QuantumCircuit:
    """Prepares (|000> + |111> + |122>)/sqrt3 in the local encoding with 5 CNOTs:
    RY on q0 (amplitudes 1/sqrt3, sqrt(2/3)), controlled-H q0->q1,
    q2 = q0 AND NOT q1 (two CNOTs), then copy q1->q3 and q2->q4."""
    qc = QuantumCircuit(5)
    qc.ry(2 * np.arccos(1 / np.sqrt(3)), 0)
    qc.ch(0, 1)
    qc.cx(0, 2); qc.cx(1, 2)
    qc.cx(1, 3); qc.cx(2, 4)
    return qc


def device_simulation(shots=8192, noise_margins=True) -> Dict:
    backend = FakeSherbrooke()
    ket     = ghz_family_ket(3)
    ket_reg = np.zeros(32, complex)
    for a, b, c in itertools.product(range(2), range(3), range(3)):
        ket_reg[encoded_register_index(a, b, c)] = ket[flat_index((a, b, c), DIMS)]
    qc = ghz233_circuit()
    assert np.allclose(Statevector(qc).data, ket_reg, atol=1e-12)

    tc   = transpile(qc, backend, optimization_level=3, seed_transpiler=RANDOM_SEED)
    phys = list(tc.layout.final_index_layout())
    tc_dm = tc.copy(); tc_dm.save_density_matrix(qubits=phys)
    sim_dm = AerSimulator.from_backend(backend, method="density_matrix")
    rho_reg = np.asarray(sim_dm.run(tc_dm, seed_simulator=RANDOM_SEED)
                         .result().data()["density_matrix"].data)

    qc_m = qc.copy(); qc_m.measure_all()
    tcm  = transpile(qc_m, backend, optimization_level=3, seed_transpiler=RANDOM_SEED)
    counts = (AerSimulator.from_backend(backend)
              .run(tcm, shots=shots, seed_simulator=RANDOM_SEED).result().get_counts())

    # device parameters actually used (initial layout of the prepared qubits)
    init = list(tc.layout.initial_index_layout())[:5]
    t1 = [backend.qubit_properties(q).t1 * 1e6 for q in init]
    t2 = [backend.qubit_properties(q).t2 * 1e6 for q in init]
    ro = [backend.target["measure"][(q,)].error for q in init]
    ecr_err = [inst.error for qargs, inst in backend.target["ecr"].items()
               if inst is not None and set(qargs) <= set(init)]

    # party-ordered 2x4x4 state and the post-selected 2x3x3 state
    rho_party = register_to_party_order(rho_reg)
    keep = [16 * a + 4 * b + c for a, b, c in itertools.product(range(2), range(3), range(3))]
    rho_proj  = rho_party[np.ix_(keep, keep)]
    p_post    = float(np.real(np.trace(rho_proj)))
    rho_post  = rho_proj / p_post
    ket_party = np.zeros(32); ket_party[keep] = ket
    F_reg  = float(np.real(ket_party @ rho_party @ ket_party))
    F_post = float(np.real(ket @ rho_post @ ket))

    W3 = verify_witness_family(3)["W_numeric"]           # Tr W = 1, exact decomposable
    wit_post = float(np.real(np.trace(W3 @ rho_post)))
    wit_reg  = p_post * wit_post                         # embedded witness on the register
    if noise_margins:                                   # the 2x4x4 SDP dominates the runtime
        q_post = ppt_mixture_noise_threshold(rho_post, DIMS)
        q_reg  = ppt_mixture_noise_threshold(rho_party, [2, 4, 4])
    else:
        q_post = q_reg = float("nan")
    return dict(counts=counts, shots=shots, backend=backend.name,
                depth=tc.depth(), n_ecr=tc.count_ops().get("ecr", 0),
                layout=init, T1_us=t1, T2_us=t2, readout_err=ro, ecr_err=ecr_err,
                F_reg=F_reg, F_post=F_post, p_post=p_post,
                witness_post=wit_post, witness_reg=wit_reg,
                noise_margin_post=q_post, noise_margin_reg=q_reg,
                gme_fidelity_reg=F_reg > 2 / 3, gme_ppt_mixture_reg=q_reg > 0)

# ============================================================================
# figures.py
# ============================================================================
"""Publication figures (vector PDF + 300-dpi PNG; legends outside axes;
fixed per-cut colours from the Okabe-Ito palette)."""
import itertools
import os

import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator


matplotlib.rcParams.update({
    "font.family"    : "serif",
    "font.size"      : 9,
    "axes.labelsize" : 9,
    "axes.titlesize" : 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "figure.dpi"     : 150,
    "savefig.dpi"    : 600,
    "axes.linewidth" : 0.6,
    "lines.linewidth": 1.2,
    "pdf.fonttype"   : 42,
    "ps.fonttype"    : 42,
})

CB = dict(
    blue="#0072B2", orange="#E69F00", green="#009E73",
    red="#D55E00",  purple="#CC79A7", teal="#56B4E9",
    yellow="#F0E442", black="#000000",
)
CUT_COLOR = {"A|BC": CB["blue"], "AB|C": CB["orange"], "AC|B": CB["green"]}


def _save(fig, fig_dir, name):
    if fig_dir:
        os.makedirs(fig_dir, exist_ok=True)
        fig.savefig(os.path.join(fig_dir, name + ".pdf"), bbox_inches="tight")
        fig.savefig(os.path.join(fig_dir, name + ".png"), bbox_inches="tight", dpi=300)
    if not matplotlib.is_interactive():      # script run: free the canvas
        plt.close(fig)


def _clean(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)


def fig_ppt_spectra(spectra, fig_dir=None):
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 3.3), layout="constrained")
    for ax, (lbl, eigs) in zip(axes, spectra.items()):
        se = np.sort(eigs); ix = np.arange(len(se))
        ax.bar(ix, se, color=CUT_COLOR[lbl], alpha=0.55, width=0.8)
        neg = se < -1e-12
        ax.bar(ix[neg], se[neg], color=CB["red"], alpha=0.85, width=0.8)
        ax.axhline(0, color="black", lw=0.7)
        ax.set_title(fr"${lbl}$   ($\min\lambda={se[0]:.4f}$)")
        ax.set_xlabel("Index"); ax.set_ylabel(r"$\lambda$")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True)); _clean(ax)
    handles = [mpatches.Patch(color=CB["red"], alpha=0.85, label=r"negative eigenvalues (NPT)"),
               mpatches.Patch(color="0.55", alpha=0.55, label="non-negative eigenvalues")]
    fig.legend(handles=handles, loc="outside lower center", ncol=2, frameon=False)
    _save(fig, fig_dir, "phase1"); plt.show(); return fig


def fig_thresholds(wit, family, fig_dir=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.2, 3.8), layout="constrained")
    p = np.linspace(0, 1, 401)
    c = float(wit[3]["c_normalised"]); D = 18
    cF = (2 / 3 - 1) / (12 - 1)                               # W_F / Tr W_F, Tr W_F = 11
    ax1.plot(p * 100, (1 - p) * c + p / D, color=CB["blue"],
             label=r"decomposable witness $W_3$ (exact)")
    ax1.plot(p * 100, (1 - p) * cF + p / D, color=CB["orange"], ls="--",
             label=r"fidelity witness $W_F$ ($\alpha=2/3$)")
    ax1.axhline(0, color="black", lw=0.6)
    ps, pF = float(wit[3]["p_star"]), 6 / 17
    ax1.axvline(ps * 100, color=CB["blue"], lw=0.8, ls=":",
                label=fr"$p^*=12/(16+\sqrt{{2}})\approx{ps*100:.2f}\%$")
    ax1.axvline(pF * 100, color=CB["orange"], lw=0.8, ls=":",
                label=fr"$p^*_F=6/17\approx{pF*100:.2f}\%$")
    ax1.set_xlabel(r"white-noise weight $p$ (%)")
    ax1.set_ylabel(r"$\mathrm{Tr}(W\rho(p))/\mathrm{Tr}\,W$")
    ax1.set_title(r"(a) $2\times3\times3$ state: witness expectation"); _clean(ax1)

    ds = np.array(sorted(family))
    dd = np.linspace(2, ds.max(), 200)
    ax2.plot(dd, 2 * dd * (dd - 1) / (2 * (dd ** 2 - 1) + np.sqrt(dd - 1)), color=CB["blue"],
             label=r"$p^*(d)$ from $W_d$ (proven lower bound)")
    ax2.plot(ds, [family[d] for d in ds], "o", color=CB["blue"], mfc="white", ms=6,
             label="PPT-mixture SDP optimum")
    ax2.plot(dd, 2 * dd / (2 * dd ** 2 - 1), color=CB["orange"], ls="--",
             label=r"$p^*_F(d)=2d/(2d^2-1)$")
    ax2.set_xlabel(r"local dimension $d$"); ax2.set_ylabel(r"threshold $p^*$")
    ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2.set_title(r"(b) $2\times d\times d$ family"); _clean(ax2)
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    fig.legend(h1 + h2, l1 + l2, loc="outside lower center", ncol=3, frameon=False)
    _save(fig, fig_dir, "phase2"); plt.show(); return fig


def fig_dynamics(times, per_cut, esd, t_ng, ng, tau_gme, params, fig_dir=None):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.2, 6.0), sharex=True,
                                   layout="constrained")
    for lbl, arr in per_cut.items():
        ax1.plot(times, arr, color=CUT_COLOR[lbl], ls="--" if lbl == "AC|B" else "-",
                 label=fr"$\mathcal{{N}}_{{{lbl}}}$  ($\tau_{{\rm ESD}}={esd[lbl]:.3f}$)")
        ax1.axvline(esd[lbl], color=CUT_COLOR[lbl], lw=0.7, ls=":")
    ax1.set_ylabel(r"negativity $\mathcal{N}_k(t)$")
    g1, gp, gc = params
    ax1.set_title(fr"Lindblad dynamics ($\gamma_1={g1}$, $\gamma_\phi={gp}$, "
                  fr"$\gamma_{{\rm corr}}={gc}$)"); _clean(ax1)
    ax2.plot(t_ng, ng, color="black", marker="o", ms=2.5,
             label=r"genuine multipartite negativity $\mathcal{N}_g(t)$")
    ax2.axvline(tau_gme, color=CB["red"], lw=0.9, ls="--",
                label=fr"$\tau_{{\rm GME}}={tau_gme:.3f}$ (PPT-mixture boundary)")
    ax2.set_xlabel(r"time $t$ (arb. units)"); ax2.set_ylabel(r"$\mathcal{N}_g(t)$")
    ax2.set_ylim(bottom=-0.005); _clean(ax2)
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    fig.legend(h1 + h2, l1 + l2, loc="outside lower center", ncol=2, frameon=False)
    _save(fig, fig_dir, "phase3"); plt.show(); return fig


def fig_gamma_sweep(sweep, fig_dir=None):
    gcs = sorted(sweep)
    fig, ax = plt.subplots(figsize=(6.2, 3.6), layout="constrained")
    for lbl in CUTS:
        ax.plot(gcs, [sweep[g]["esd"][lbl] for g in gcs], marker="o", ms=4,
                color=CUT_COLOR[lbl], ls="-" if lbl != "AC|B" else "--",
                label=fr"$\tau_{{\rm ESD}}({lbl})$")
    ax.plot(gcs, [sweep[g]["tau_gme"] for g in gcs], marker="s", ms=4, color="black",
            label=r"$\tau_{\rm GME}$ (PPT-mixture criterion)")
    ax.set_xlabel(r"correlated dephasing rate $\gamma_{\rm corr}$")
    ax.set_ylabel(r"lifetime"); _clean(ax)
    fig.legend(loc="outside lower center", ncol=2, frameon=False)
    _save(fig, fig_dir, "phase3b"); plt.show(); return fig


def fig_device(hw, fig_dir=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.9), width_ratios=[2.2, 1],
                                   layout="constrained")
    shots = hw["shots"]; probs = {}
    leak = 0
    for key, cnt in hw["counts"].items():
        k = int(key.replace(" ", ""), 2)
        q = [(k >> i) & 1 for i in range(5)]
        b, c = q[1] + 2 * q[2], q[3] + 2 * q[4]
        if b == 3 or c == 3:
            leak += cnt; continue
        probs[(q[0], b, c)] = probs.get((q[0], b, c), 0) + cnt / shots
    labels = ["".join(map(str, s)) for s in itertools.product(range(2), range(3), range(3))]
    vals = [probs.get(tuple(int(x) for x in s), 0) for s in labels] + [leak / shots]
    cols = [CB["blue"] if s in ("000", "111", "122") else "0.6" for s in labels] + [CB["red"]]
    ax1.bar(range(len(vals)), vals, color=cols, width=0.75)
    ax1.set_xticks(range(len(vals)))
    ax1.set_xticklabels(labels + ["leak"], rotation=60, fontsize=6.5)
    ax1.set_ylabel("probability"); ax1.set_xlabel(r"measured $|abc\rangle$")
    ax1.set_title(f"{hw['shots']} shots, {hw['backend']} noise model"); _clean(ax1)
    ax2.axis("off")
    rows = [("register fidelity $F$", f"{hw['F_reg']:.4f}"),
            ("post-selected fidelity", f"{hw['F_post']:.4f}"),
            ("post-selection prob.", f"{hw['p_post']:.4f}"),
            (r"$\mathrm{Tr}(W_3\rho)$ (register)", f"{hw['witness_reg']:+.4f}"),
            ("extra white noise (register)", f"{hw['noise_margin_reg']:.3f}"),
            ("ECR gates / depth", f"{hw['n_ecr']} / {hw['depth']}")]
    for i, (k, v) in enumerate(rows):
        ax2.text(0.0, 0.95 - 0.14 * i, k, transform=ax2.transAxes, fontsize=8, color="0.3")
        ax2.text(1.0, 0.95 - 0.14 * i, v, transform=ax2.transAxes, fontsize=8,
                 ha="right", fontweight="bold")
    handles = [mpatches.Patch(color=CB["blue"], label="target components"),
               mpatches.Patch(color="0.6", label="other code states"),
               mpatches.Patch(color=CB["red"], label="leakage (qutrit level 3)")]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False)
    _save(fig, fig_dir, "phase4"); plt.show(); return fig

import types as _types
figures = _types.SimpleNamespace(**{k: v for k, v in list(globals().items()) if k.startswith('fig_')})

# ============================================================================
# pipeline.py
# ============================================================================
"""The reproducibility pipeline, split into timed phases.

Phase  Content                                                       Paper
0      fidelity witness, alpha = 2/3, p*_F = 6/17                    Sec. III
1      partial-transpose spectra                                     Fig. 1
2      exact witness family, exact primal certificate, SDPs          Sec. IV-V, Fig. 2, Table 1
3      Lindblad dynamics, ESD times, N_g(t), GME lifetime            Sec. VII, Fig. 3
3b     gamma_corr sweep                                              Sec. VII-D, Fig. 4
4      noisy-device simulation (FakeSherbrooke)                      Sec. VIII, Fig. 5, Table 3
5      re-analysis of the Malik et al. (2016) photonic data          Sec. VI, Table 2
"""
import time
from dataclasses import dataclass, field
from typing import Dict, Iterable, Tuple

import numpy as np


ALL_PHASES = ("0", "1", "2", "3", "3b", "4", "5")


@dataclass
class Settings:
    gamma_1: float = 0.05
    gamma_phi: float = 0.10
    gamma_corr: float = 0.02
    t_max: float = 30.0
    gamma_corr_sweep: Tuple[float, ...] = (0.0, 0.005, 0.01, 0.02, 0.04, 0.08)
    family_dims: Tuple[int, ...] = (2, 3, 4, 5, 6)
    dual_dims: Tuple[int, ...] = (2, 3, 4, 5, 6, 7, 8)
    ng_points: int = 41
    shots: int = 8192
    noise_margins: bool = True
    fig_dir: str = "results/figures"

    @classmethod
    def quick(cls, **kw):
        """Smoke-test settings: same code paths, fewer points (minutes, not tens of minutes)."""
        return cls(gamma_corr_sweep=(0.0, 0.08), family_dims=(2, 3, 4), dual_dims=(2, 3, 4), ng_points=6,
                   shots=2048, noise_margins=False, **kw)


@dataclass
class Context:
    settings: Settings
    out: Dict = field(default_factory=dict)
    timings: Dict = field(default_factory=dict)

    def __post_init__(self):
        self.ket = ghz_family_ket(3)
        self.rho0 = np.outer(self.ket, self.ket)


def sep(title: str, width: int = 72):
    bar = "=" * width
    print(f"\n{bar}\n  {title}\n{bar}")


def phase0(ctx: Context):
    sep("Phase 0 - Fidelity witness")
    ket, D = ctx.ket, 18
    alphas = {}
    for lbl, k in CUTS.items():
        rest = [p for p in range(3) if p not in k]
        M = ket.reshape(DIMS).transpose(list(k) + rest).reshape(int(np.prod([DIMS[p] for p in k])), -1)
        alphas[lbl] = float(np.linalg.svd(M, compute_uv=False)[0] ** 2)
    alpha = max(alphas.values()); pF = D * (1 - alpha) / (D - 1)
    print("  largest squared Schmidt coefficient per cut:",
          {k: round(v, 10) for k, v in alphas.items()})
    print(f"  alpha = {alpha:.10f} (2/3)   p*_F = D(1-alpha)/(D-1) = {pF:.10f} (6/17)")
    ctx.out.update(alpha=alpha, pF=pF)


def phase1(ctx: Context):
    sep("Phase 1 - Partial-transpose spectra")
    spectra = {lbl: np.linalg.eigvalsh(partial_transpose(ctx.rho0, DIMS, k)) for lbl, k in CUTS.items()}
    for lbl, ev in spectra.items():
        print(f"  {lbl}: min eigenvalue = {ev[0]:+.8f}   negativity = {-ev[ev < 0].sum():.8f}")
    figures.fig_ppt_spectra(spectra, ctx.settings.fig_dir)


def phase2(ctx: Context):
    sep("Phase 2 - PPT-mixture criterion: exact witness, exact primal, SDP")
    s, rho0 = ctx.settings, ctx.rho0
    wit = {}
    for d in sorted(set((2, 3, 4, 5, 6)) | set(s.family_dims)):
        wit[d] = v = verify_witness_family(d)
        print(f"  d={d}: W_d = P_k + T_k(Q_k) exact: {v['decomposition_exact']}, "
              f"P_k,Q_k PSD exact: {v['P_Q_psd_exact']}, Tr W~ = {v['trace_W']}, "
              f"Tr(W~ rho_psi) = {v['tr_W_rho_psi']}, p* = {float(v['p_star']):.10f}")
    print(f"  d=3: c = Tr(W rho_psi)/Tr W = {wit[3]['c_normalised']}  (= -(4-sqrt2)/21)")
    print(f"       p* = {wit[3]['p_star']}  (= 12/(16+sqrt2) = 6(16-sqrt2)/127)")
    primal = exact_primal_certificate_d3()
    print(f"  exact primal PPT-mixture decomposition at p*: sum exact = {primal['sum_equals_rho']}, "
          f"all six PSD conditions exact = {primal['all_ok']}, "
          f"complementary slackness = {primal['complementary_slackness']}")
    q_num = ppt_mixture_noise_threshold(rho0, DIMS)
    c_num, _ = decomposable_witness_value(rho0, DIMS)
    print(f"  numerical primal threshold = {q_num:.10f}; numerical dual c_min = {c_num:.10f}")
    pit = weak_constraint_pitfall(rho0, DIMS)
    print(f"  pitfall check: program with only W + T_k(W) >= 0 gives {pit['value']:.6f} "
          f"(naive threshold {pit['naive_threshold']:.4f}) but its optimiser has "
          f"expectation {pit['value_on_product_state']:+.6f} on a product state -> not a witness")
    family = {}
    for d in s.family_dims:
        family[d] = family_threshold_sdp(d)
        print(f"  family d={d}: SDP threshold {family[d]:.10f}   W_d formula "
              f"{float(wit[d]['p_star']):.10f}   diff {family[d] - float(wit[d]['p_star']):+.1e}")
    D = lambda d: 2 * d * d
    for d in s.family_dims:                         # optimal c_min implied by the primal threshold
        c_from_primal = -family[d] / (D(d) * (1 - family[d]))
        print(f"  family d={d}: c_min implied by primal = {c_from_primal:.6f}   "
              f"W_d value Tr(W_d rho)/Tr W_d = {float(wit[d]['c_normalised']) if d in wit else float('nan'):.6f}")
    dual = {}
    for d in s.dual_dims:
        dual[d] = r = family_witness_sdp(d)
        pw = 2 * d * (d - 1) / (2 * (d * d - 1) + (d - 1) ** 0.5)
        cw = -2 * (d - 1) / d / (4 * (d - 1) + 2 * (d - 1) ** 0.5)
        print(f"  family dual d={d}: c_min = {r['c_min']:.6f} (W_d: {cw:.6f})   threshold {r['threshold']:.6f} "
              f"(W_d formula {pw:.6f}, diff {r['threshold'] - pw:+.1e})")
    figures.fig_thresholds(wit, family, s.fig_dir)
    ctx.out.update(witness=wit, primal=primal, q_num=q_num, c_num=c_num, pitfall=pit, family=family,
                   family_dual=dual)


def phase3(ctx: Context):
    import qutip as qt
    sep("Phase 3 - Lindblad dynamics")
    s, rho0, D = ctx.settings, ctx.rho0, 18
    L = liouvillian(DIMS, s.gamma_1, s.gamma_phi, s.gamma_corr)
    prop = Propagator(L, rho0.astype(complex))
    drho = (L @ rho0.reshape(-1)).reshape(D, D)
    rates = {}
    for lbl, k in CUTS.items():
        rates[lbl] = negativity_right_derivative(rho0, drho, DIMS, k)
        neg_term, ker_term = negativity_rate_terms(rho0, drho, DIMS, k)
        h = 1e-7
        fd = (negativity(prop(h), DIMS, k) - negativity(rho0, DIMS, k)) / h
        print(f"  dN/dt|0+ {lbl}: {rates[lbl]:+.6f}   (finite difference {fd:+.6f}); "
              f"negative-space term {neg_term:+.6f}, kernel term {ker_term:+.6f}, "
              f"overstatement without kernel {abs(neg_term) / abs(rates[lbl]) - 1:.1%}")
    times = np.linspace(0, 14, 561)
    per_cut = {lbl: np.array([negativity(prop(t), DIMS, k) for t in times]) for lbl, k in CUTS.items()}
    mono = {lbl: bool(np.max(np.diff(a)) <= 1e-12) for lbl, a in per_cut.items()}
    esd = {lbl: esd_time(prop, DIMS, k, s.t_max) for lbl, k in CUTS.items()}
    print("  monotone per-cut decay on 561-point grid:", mono)
    print("  exact ESD times:", {k: round(v, 4) for k, v in esd.items()})
    qt_ops = [qt.Qobj(C, dims=[DIMS, DIMS])
              for C in collapse_operators(DIMS, s.gamma_1, s.gamma_phi, s.gamma_corr)]
    t_chk = np.linspace(0, 10, 11)
    res = qt.mesolve(qt.Qobj(np.zeros((D, D)), dims=[DIMS, DIMS]),
                     qt.Qobj(rho0, dims=[DIMS, DIMS]), t_chk, qt_ops,
                     options={"atol": 1e-12, "rtol": 1e-10, "nsteps": 100000})
    dev = max(np.abs(st.full() - prop(t)).max() for st, t in zip(res.states, t_chk))
    print(f"  QuTiP mesolve cross-check: max |rho_qutip - rho_exact| = {dev:.1e}")
    t_ng = np.linspace(0, 10, s.ng_points)
    ng = np.array([genuine_multipartite_negativity(prop(t), DIMS, real=True) for t in t_ng])
    tau = gme_lifetime(prop, DIMS, esd["A|BC"])
    n_min = min(negativity(rho0, DIMS, k) for k in CUTS.values())
    print(f"  N_g(0) = {ng[0]:.6f}   (bound min_k N_k = {n_min:.6f})   "
          f"GME lifetime (PPT-mixture boundary) tau_GME = {tau:.4f}")
    t_lo, t_hi = round(tau, 2) - 0.01, round(tau, 2) + 0.01
    v_lo = decomposable_witness_value(prop(t_lo), DIMS, real=True)[0]
    v_hi = decomposable_witness_value(prop(t_hi), DIMS, real=True)[0]
    print(f"  tau_GME bracket: witness value {v_lo:+.1e} at t = {t_lo:.2f}, {v_hi:+.1e} at t = {t_hi:.2f}")
    figures.fig_dynamics(times, per_cut, esd, t_ng, ng, tau,
                         (s.gamma_1, s.gamma_phi, s.gamma_corr), s.fig_dir)
    ctx.out.update(rates=rates, esd=esd, monotone=mono, qutip_dev=dev, ng=(t_ng, ng), tau_gme=tau)


def phase3b(ctx: Context):
    sep("Phase 3b - gamma_corr sweep")
    s, rho0 = ctx.settings, ctx.rho0
    sweep = {}
    for gc in s.gamma_corr_sweep:
        pr = Propagator(liouvillian(DIMS, s.gamma_1, s.gamma_phi, gc), rho0.astype(complex))
        e = {lbl: esd_time(pr, DIMS, k, s.t_max) for lbl, k in CUTS.items()}
        sweep[gc] = dict(esd=e, tau_gme=gme_lifetime(pr, DIMS, e["A|BC"]))
        print(f"  gamma_corr={gc:.3f}: ESD A|BC={e['A|BC']:.4f}  AB|C={e['AB|C']:.4f}  "
              f"AC|B={e['AC|B']:.4f}   tau_GME={sweep[gc]['tau_gme']:.4f}")
    g0, g1 = min(sweep), max(sweep)
    if g1 > g0:
        red = {lbl: 1 - sweep[g1]["esd"][lbl] / sweep[g0]["esd"][lbl] for lbl in CUTS}
        red["tau_GME"] = 1 - sweep[g1]["tau_gme"] / sweep[g0]["tau_gme"]
        print(f"  reduction from gamma_corr={g0} to {g1}: " +
              ", ".join(f"{k} {v:.1%}" for k, v in red.items()))
    figures.fig_gamma_sweep(sweep, s.fig_dir)
    ctx.out["sweep"] = sweep


def phase4(ctx: Context):
    sep("Phase 4 - Noisy-device simulation (FakeSherbrooke, local encoding)")
    hw = device_simulation(ctx.settings.shots, ctx.settings.noise_margins)
    print(f"  backend {hw['backend']}, physical qubits {hw['layout']}, "
          f"ECR count {hw['n_ecr']}, depth {hw['depth']}")
    print(f"  T1 (us) {np.round(hw['T1_us'], 1)}  T2 (us) {np.round(hw['T2_us'], 1)}")
    print(f"  readout errors {np.round(hw['readout_err'], 4)}  ECR errors {np.round(hw['ecr_err'], 4)}")
    print(f"  register fidelity F = {hw['F_reg']:.4f} (> 2/3: {hw['gme_fidelity_reg']})")
    print(f"  post-selection probability = {hw['p_post']:.4f}, post-selected fidelity = {hw['F_post']:.4f}")
    print(f"  Tr(W_3 rho) register = {hw['witness_reg']:+.5f}, post-selected = {hw['witness_post']:+.5f}")
    print(f"  extra white noise tolerated before PPT mixture: register {hw['noise_margin_reg']:.4f}, "
          f"post-selected {hw['noise_margin_post']:.4f}")
    figures.fig_device(hw, ctx.settings.fig_dir)
    ctx.out["device"] = hw


def phase5(ctx: Context):
    sep("Phase 5 - Re-analysis of the photonic (3,3,2) experiment (Malik et al. 2016)")
    m = malik2016_certificate()
    ph = m["coherence_phases_deg"]
    print(f"  coherence phases (deg): {ph[0]:.1f}, {ph[1]:.1f}, {ph[2]:.1f}; moduli |rho_000,111| = "
          f"{m['coherence_moduli'][0]:.3f}, |rho_000,122| = {m['coherence_moduli'][1]:.3f}")
    print(f"  fidelity with the two GHZ coherences aligned = {m['F']:.4f}  (reported 0.801 +- 0.018); "
          f"residual phase of rho_111,122 = {m['residual_phase_deg']:.2f} deg")
    print(f"  positivity bound on the unmeasured-coherence term: {m['g_term_bound']:.4f}")
    print(f"  Tr(W~3 rho_exp) <= {m['witness_unnorm']:.4f}   normalised Tr(W3 rho_exp) <= {m['witness']:.4f}")
    print("  worst case if every element is shifted by delta: " +
          ", ".join(f"delta={dl}: {v:+.4f}" for dl, v in m["witness_worst_case"].items()))
    print(f"  extra white noise tolerated: witness {m['q_witness']:.3f}, fidelity test {m['q_fidelity']:.3f}")
    print(f"  witness negative unless |rho_000,111| + |rho_000,122| < {m['coherence_sum_needed']:.4f} "
          f"(measured {m['coherence_sum']:.4f})")
    ctx.out["malik2016"] = m


PHASES = {"0": phase0, "1": phase1, "2": phase2, "3": phase3, "3b": phase3b,
          "4": phase4, "5": phase5}


def run_pipeline(phases: Iterable[str] = ALL_PHASES, settings: Settings = None) -> Context:
    """Run the selected phases in order and record the wall-clock time of each."""
    ctx = Context(settings or Settings())
    for name in phases:
        t0 = time.perf_counter()
        PHASES[name](ctx)
        ctx.timings[name] = time.perf_counter() - t0
    sep("Timing (wall clock)")
    for name, sec in ctx.timings.items():
        print(f"  phase {name:>2}: {sec:8.1f} s")
    print(f"  total   : {sum(ctx.timings.values()):8.1f} s")
    return ctx

# ============================================================================
# Entry point
# ============================================================================
ctx = run_pipeline(settings=Settings(fig_dir="./figures"))
